# Paper 2 — Rebuild RF v1 Training Table from Raw Sources

**SCRIPT: Paper2_RF_v1_Rebuild_TrainingTable.ipynb**

**Purpose:** the original RF v1 training table (the one that produced R²=0.554, Table 7,
Figure 4/5) is no longer available. This notebook rebuilds it from the same raw sources
described in Section 2.5/3.5 of the manuscript, so that `Paper2_RF_v2_Additional_Covariates.ipynb`
has something to load.

**Run this notebook FIRST**, then feed its output (`rf_v1_training_table.csv`) into the v2
notebook's Step 1 upload cell.

**Data sources reconstructed here (per Figure 4's 17 predictors):**

| Feature | Source |
|---|---|
| PM2.5, PM10 (target) | Air4Thai API, stations 24T/25T |
| AOD (0.47µm, 0.55µm) | MODIS MAIAC (MCD19A2), via Google Earth Engine |
| Temperature, wind speed 10m/50m, wind direction, relative humidity, rainfall, surface pressure | NASA POWER daily point API |
| Fire count, distance to nearest fire | NASA FIRMS active-fire archive |
| NDVI, EVI | MODIS MOD13Q1 (16-day, 250m), via Google Earth Engine |
| Distance to nearest factory / nearest high-risk factory | Reused from existing DIW geocoding (384 facilities / 38 high-risk subset) — **not rebuilt here** |
| Month, day of week | Derived from date |

**Before running:** fill in every `TODO` — especially the exact 24T/25T coordinates and your
FIRMS `MAP_KEY` (free, from https://firms.modaps.eosdis.nasa.gov/api/map_key/).


In [ ]:
# SECTION: Setup & defensive imports
import sys, subprocess

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _imp in [("geopandas", "geopandas"), ("requests", "requests")]:
    _ensure(_pkg, _imp)

import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta


## Step 0 — Station coordinates

Confirmed from the Air4Thai official real-time API (cross-checked across multiple mirrors):
- **24T** — Na Phra Lan Police Station: 14.685833, 100.871996
- **25T** — Khao Noi Fire Station, Pak Phriao: 14.526343, 100.926047


In [ ]:
# SECTION: Station metadata (confirmed from Air4Thai official API, cross-checked
# across multiple sources)
STATIONS = {
    "24T": {"name": "Na Phra Lan Police Station", "lat": 14.685833, "lon": 100.871996},
    "25T": {"name": "Khao Noi Fire Station, Pak Phriao", "lat": 14.526343, "lon": 100.926047},
}

DATE_START = "2016-01-01"
DATE_END = "2025-12-31"


## Step 1 — Load PM2.5 / PM10 (already prepared)

Ground-truth PM2.5/PM10 for 24T/25T (2016–2025) has already been extracted and verified
against the original manuscript's Table 1/2/3 (values matched almost exactly — see prior
analysis). Just upload **`pm25_pm10_24T_25T_ready.csv`** below.


In [ ]:
# SECTION: Load prepared PM2.5/PM10 data
from google.colab import files
import os

if os.path.exists("pm25_pm10_24T_25T_ready.csv"):
    pm_df = pd.read_csv("pm25_pm10_24T_25T_ready.csv", parse_dates=["date"])
else:
    print("Upload pm25_pm10_24T_25T_ready.csv:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    pm_df = pd.read_csv(fname, parse_dates=["date"])

print(pm_df.shape)
pm_df.head()


## Step 2 — Earth Engine setup (AOD + NDVI/EVI)


In [ ]:
# SECTION: Earth Engine auth
_ensure("earthengine-api", "ee")
import ee

EE_PROJECT_ID = "saraburi-thesis"  # confirmed Cloud Project ID

try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
print("Earth Engine ready, project:", EE_PROJECT_ID)


In [ ]:
# SECTION: AOD (MODIS MAIAC) per station per date
def get_daily_aod(lat, lon, date_start, date_end):
    point = ee.Geometry.Point([lon, lat])
    coll = (ee.ImageCollection("MODIS/061/MCD19A2_GRANULES")
            .filterDate(date_start, date_end)
            .filterBounds(point)
            .select(["Optical_Depth_047", "Optical_Depth_055"]))

    def extract(img):
        val = img.reduceRegion(ee.Reducer.mean(), point, 1000)
        return ee.Feature(None, {
            "date": img.date().format("YYYY-MM-dd"),
            "aod_047": val.get("Optical_Depth_047"),
            "aod_055": val.get("Optical_Depth_055"),
        })

    fc = coll.map(extract)
    result = fc.getInfo()
    rows = [f["properties"] for f in result["features"]]
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"])
    # MAIAC scale factor is 0.001
    for c in ["aod_047", "aod_055"]:
        df[c] = pd.to_numeric(df[c], errors="coerce") * 0.001
    return df.groupby("date", as_index=False).mean(numeric_only=True)

aod_frames = []
for sid, meta in STATIONS.items():
    df_aod = get_daily_aod(meta["lat"], meta["lon"], DATE_START, DATE_END)
    df_aod["station"] = sid
    aod_frames.append(df_aod)

aod_df = pd.concat(aod_frames, ignore_index=True) if aod_frames else pd.DataFrame()
print(aod_df.shape)
aod_df.head()


In [ ]:
# SECTION: NDVI / EVI (MODIS MOD13Q1, 16-day) per station per date
def get_vi_series(lat, lon, date_start, date_end):
    point = ee.Geometry.Point([lon, lat])
    coll = (ee.ImageCollection("MODIS/061/MOD13Q1")
            .filterDate(date_start, date_end)
            .filterBounds(point)
            .select(["NDVI", "EVI"]))

    def extract(img):
        val = img.reduceRegion(ee.Reducer.mean(), point, 250)
        return ee.Feature(None, {
            "date": img.date().format("YYYY-MM-dd"),
            "ndvi": val.get("NDVI"),
            "evi": val.get("EVI"),
        })

    fc = coll.map(extract)
    result = fc.getInfo()
    rows = [f["properties"] for f in result["features"]]
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"])
    # MOD13Q1 scale factor is 0.0001
    for c in ["ndvi", "evi"]:
        df[c] = pd.to_numeric(df[c], errors="coerce") * 0.0001
    return df

vi_frames = []
for sid, meta in STATIONS.items():
    df_vi = get_vi_series(meta["lat"], meta["lon"], DATE_START, DATE_END)
    df_vi["station"] = sid
    vi_frames.append(df_vi)

vi_df = pd.concat(vi_frames, ignore_index=True) if vi_frames else pd.DataFrame()
# NDVI/EVI are 16-day composites — forward-fill to daily resolution per station
vi_df = vi_df.set_index("date").groupby("station").resample("D").ffill().drop(columns="station").reset_index()
print(vi_df.shape)
vi_df.head()


## Step 3 — Meteorology from NASA POWER (free, no API key needed)

Provides temperature, wind speed at 10m and 50m, wind direction, relative humidity,
rainfall, and surface pressure as a single daily point request per station.


In [ ]:
# SECTION: NASA POWER meteorological variables
POWER_PARAMS = ["T2M", "WS10M", "WS50M", "WD10M", "RH2M", "PRECTOTCORR", "PS"]

def get_power_data(lat, lon, date_start, date_end):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": ",".join(POWER_PARAMS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": date_start.replace("-", ""),
        "end": date_end.replace("-", ""),
        "format": "JSON",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()["properties"]["parameter"]
    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df = df.reset_index().rename(columns={
        "index": "date", "T2M": "temperature", "WS10M": "wind_speed_10m",
        "WS50M": "wind_speed_50m", "WD10M": "wind_direction",
        "RH2M": "relative_humidity", "PRECTOTCORR": "rainfall", "PS": "surface_pressure",
    })
    # NASA POWER uses -999 as a missing-value sentinel
    df = df.replace(-999, np.nan)
    return df

power_frames = []
for sid, meta in STATIONS.items():
    df_power = get_power_data(meta["lat"], meta["lon"], DATE_START, DATE_END)
    df_power["station"] = sid
    power_frames.append(df_power)
    time.sleep(1)

power_df = pd.concat(power_frames, ignore_index=True)
print(power_df.shape)
power_df.head()


## Step 4 — Fire count + distance to nearest fire from NASA FIRMS

**TODO: get a free `MAP_KEY`** from https://firms.modaps.eosdis.nasa.gov/api/map_key/ and
paste it below. FIRMS archive requests are capped at 10 years per request and limited by
area — this fetches a bounding box around Saraburi once, then computes per-day count and
per-day nearest-fire distance for each station from that single pull (more efficient than
one request per station).


In [ ]:
# SECTION: NASA FIRMS active fire detections
FIRMS_MAP_KEY = "REPLACE_WITH_YOUR_FIRMS_MAP_KEY"  # TODO

# Bounding box around the Saraburi study area — TODO confirm matches your Paper 1 study extent
FIRE_BBOX = "100.55,14.25,101.45,14.95"  # west,south,east,north

def get_firms_archive(bbox, date_start, date_end, dataset="MODIS_NRT", map_key=FIRMS_MAP_KEY):
    """FIRMS archive API caps each request to 10 days for area queries on some tiers —
    this loops in chunks to be safe."""
    all_rows = []
    start = datetime.strptime(date_start, "%Y-%m-%d")
    end = datetime.strptime(date_end, "%Y-%m-%d")
    chunk_days = 10
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/"
               f"{dataset}/{bbox}/{chunk_days}/{cur.strftime('%Y-%m-%d')}")
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            from io import StringIO
            chunk_df = pd.read_csv(StringIO(r.text))
            all_rows.append(chunk_df)
        except Exception as e:
            print(f"FIRMS fetch failed for {cur.date()}: {e}")
        cur = chunk_end + timedelta(days=1)
        time.sleep(0.5)
    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# NOTE: MODIS_NRT only covers recent ~2 months; for the full 2016-2025 archive use
# dataset="MODIS_SP" (standard processing, historical) instead — TODO pick the right one
# per period, since NRT + SP together give full historical coverage.
fires_df = get_firms_archive(FIRE_BBOX, DATE_START, DATE_END, dataset="MODIS_SP")
print(fires_df.shape)
fires_df.head()


In [ ]:
# SECTION: Fire count + nearest-fire distance per station per day
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

if not fires_df.empty:
    fires_df["acq_date"] = pd.to_datetime(fires_df["acq_date"])

fire_features = []
for sid, meta in STATIONS.items():
    for date, group in fires_df.groupby("acq_date") if not fires_df.empty else []:
        dists = group.apply(lambda r: haversine_km(meta["lat"], meta["lon"], r["latitude"], r["longitude"]), axis=1)
        fire_features.append({
            "station": sid, "date": date,
            "fire_count": len(group),
            "dist_to_nearest_fire_km": dists.min() if len(dists) else np.nan,
        })

fire_daily = pd.DataFrame(fire_features)
print(fire_daily.shape)
fire_daily.head()


## Step 5 — Distance to nearest factory (reused, not rebuilt)

Uses the DIW facility geocoding you already have — no need to redo this part.


In [ ]:
# SECTION: Load existing facility distances (static per station, doesn't vary by date)
# TODO: path to your existing geocoded facility table (384 records) with lat/lon
import os
from google.colab import files

def load_or_upload(default_name):
    if os.path.exists(default_name):
        return pd.read_csv(default_name)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file uploaded — expected {default_name}")
    fname = list(uploaded.keys())[0]
    return pd.read_csv(fname)

facilities = load_or_upload("diw_facilities_geocoded.csv")

def nearest_dist_km(lat, lon, fac_df):
    dists = fac_df.apply(lambda r: haversine_km(lat, lon, r["lat"], r["lon"]), axis=1)
    return dists.min()

# TODO: if you have a separate flag/column marking which 38 are "high-risk", filter here;
# otherwise this treats all 384 the same for both distance columns until that's confirmed.
station_factory_dist = {}
for sid, meta in STATIONS.items():
    d_all = nearest_dist_km(meta["lat"], meta["lon"], facilities)
    station_factory_dist[sid] = {"dist_to_nearest_factory_km": d_all}
    # TODO: once high-risk subset criteria is confirmed, compute dist_to_risk_factory_km
    # from that subset instead of reusing d_all
    station_factory_dist[sid]["dist_to_risk_factory_km"] = d_all

print(station_factory_dist)


## Step 6 — Merge everything into the final training table


In [ ]:
# SECTION: Merge all sources into one row-per-station-day table
df = pm_df.merge(aod_df, on=["date", "station"], how="left")
df = df.merge(vi_df, on=["date", "station"], how="left")
df = df.merge(power_df, on=["date", "station"], how="left")
df = df.merge(fire_daily, on=["date", "station"], how="left")

df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

for sid, dists in station_factory_dist.items():
    mask = df["station"] == sid
    df.loc[mask, "dist_to_nearest_factory_km"] = dists["dist_to_nearest_factory_km"]
    df.loc[mask, "dist_to_risk_factory_km"] = dists["dist_to_risk_factory_km"]

# Attach lat/lon so the v2 notebook can compute LST / road-distance / land-use per row
for sid, meta in STATIONS.items():
    mask = df["station"] == sid
    df.loc[mask, "lat"] = meta["lat"]
    df.loc[mask, "lon"] = meta["lon"]

df = df.rename(columns={"pm25": "pm25", "pm10": "pm10"})

print(f"Final table: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values per column:")
print(df.isna().mean().sort_values(ascending=False).round(3))
df.head()


In [ ]:
# SECTION: Export for use in Paper2_RF_v2_Additional_Covariates.ipynb
df.to_csv("rf_v1_training_table.csv", index=False)
try:
    files.download("rf_v1_training_table.csv")
except Exception as e:
    print("Download skipped (not in Colab):", e)

print("Done — upload rf_v1_training_table.csv into the v2 notebook's Step 1 cell.")


## Important note on comparing this to the original R²=0.554

Because this table is **reconstructed**, not identical to whatever produced the original
0.554, the baseline R² you get from re-running the *original* 17-feature model on this
rebuilt table may differ slightly from 0.554 — possibly due to: API data revisions since the
original pull, slightly different date ranges, or rounding/processing differences. **Run the
baseline (17-feature, no new covariates) model on this table first** and report whichever
number it actually gives as the "v1 baseline (reconstructed)" in the revised manuscript,
rather than assuming it will reproduce exactly 0.554. This is worth a one-sentence footnote
in the manuscript for transparency.
